# Train wav2vec2 Quran ASR + corrector — Colab (Option A, no HF Hub)

**Storage:** audio + dataset + caches on Colab `/content` (~70 GB, ephemeral); trained **checkpoints** on Google Drive.
**No Hugging Face account / Hub:** the trained model is saved to Drive only (not pushed anywhere).
Switch runtime to **GPU (T4)** before running.


## 0. Route HF *library* caches to the big /content partition (FIRST)

Not using the HF Hub as a service — this only caches the one-time base-model download (~1.2 GB)
and the `input_values` cache (~10 GB) on the big `/content` disk instead of the small ~12 GB root.


In [ ]:
import os
os.environ['HF_HOME'] = '/content/hf_cache'
os.makedirs('/content/hf_cache', exist_ok=True)

!nvidia-smi
!df -h /content | tail -1


## 1. Install dependencies


In [ ]:
!pip install -q -U pip
!pip install -q torch torchaudio transformers datasets accelerate evaluate jiwer \
    librosa soundfile torchcodec huggingface_hub pyyaml requests tqdm
!apt-get -qq install -y ffmpeg > /dev/null


## 2. Get the code
Clone from your GitHub fork, or upload the project folder.


In [ ]:
# !git clone https://github.com/<you>/model-asr-quran.git /content/model-asr-quran
%cd /content/model-asr-quran
!pip install -q -e .
!python scripts/cloud_preflight.py


## 3. Mount Google Drive (for checkpoints)
Audio stays on `/content`; Drive holds only the trained model.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
os.makedirs('/content/drive/MyDrive/quran_asr/checkpoints', exist_ok=True)

# If Drive is FULL from a previous run, free it:
# !rm -rf /content/drive/MyDrive/quran_asr/data /content/drive/MyDrive/quran_asr/processed


## 4. Download audio (to /content, resume-safe, EPHEMERAL)
~12k files (2 reciters). ~30-40 min. Add `--no-convert` to keep MP3 (~half size) if `/content` is tight.


In [ ]:
!python scripts/download.py --config configs/cloud_local.yaml --max-workers 16 --qps 30


## 5. Build dataset + vocab


In [ ]:
!python scripts/build.py --config configs/cloud_local.yaml
!python scripts/build_vocab.py --config configs/cloud_local.yaml


## 6. Train
Checkpoints save to **Drive** (`logging.output_dir`). No HF Hub push.


In [ ]:
from quran_asr.config import Config
from quran_asr.training.train import train

cfg = Config.from_yaml('configs/cloud_local.yaml')   # logging.hub_repo stays null = no push
trainer, processor = train(cfg)
print('training done')


## 7. Evaluate


In [ ]:
!python scripts/evaluate.py --config configs/cloud_local.yaml --split test \
    --out /content/quran_asr/metrics.json


## 8. Corrector demo


In [ ]:
!python scripts/correct.py --model-dir /content/drive/MyDrive/quran_asr/checkpoints \
    --audio /content/quran_asr/data/raw/audio/Husary_128kbps_Mujawwad/001001.wav \
    --text "بِسْمِ ٱللَّهِ ٱلرَّحْمَـٰنِ ٱلرَّحِيمِ"


## Resume after disconnect
Audio on `/content` is gone; Drive checkpoints survive. Re-run cells 0→5, then train with `resume=True`.

## Out of disk on /content?
Add `--no-convert` to step 4 (MP3, ~half size, decoded on-the-fly — verified). Or train one reciter first.

## Getting your model out
The trained model + processor are in `/content/drive/MyDrive/quran_asr/checkpoints/final/`.
Copy that folder to your backend machine to serve it.
